In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
import pandas as pd
from utils import jaccard_similarity, get_jaccard_features, get_length_features

## `jaccard_similarity` – token-level Jaccard

Given two strings, the Jaccard similarity is defined as:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

where A and B are the **sets of tokens** obtained after lowercasing and splitting on non-alphanumeric characters.

The function is implemented **from scratch** without any sklearn/scipy helper.

### Why Jaccard?
- It is insensitive to document length (unlike raw word overlap counts).
- It has a natural interpretation: 0 = no shared words, 1 = identical token sets.
- Duplicate questions tend to share most of their vocabulary → high Jaccard.

In [2]:
# Perfect duplicates
print(jaccard_similarity("What is machine learning?", "What is machine learning?"))   # 1.0

# Near-duplicates (same intent, slightly different wording)
print(jaccard_similarity("What is ML?", "What is machine learning?"))                 # ~0.33

# Totally different
print(jaccard_similarity("How do I cook pasta?", "What is the capital of France?"))   # 0.0

# Edge case: empty / NaN
print(jaccard_similarity("", ""))   # 0.0

1.0
0.4
0.0
0.0


### Distribution on a sample

In [3]:
import os
import sklearn.model_selection
import numpy as np
from utils import get_data_splits

DATA_PATH = os.path.join(os.path.expanduser("~"), "Datasets", "QuoraQuestionPairs", "quora_data.csv")
quora_df = pd.read_csv(DATA_PATH)
train_df, val_df, test_df = get_data_splits(quora_df)

sample = train_df.sample(500, random_state=42)
X_jac = get_jaccard_features(sample)

print("Jaccard stats (sample of 500):")
print(f"  mean  = {X_jac.mean():.3f}")
print(f"  std   = {X_jac.std():.3f}")
print(f"  min   = {X_jac.min():.3f}")
print(f"  max   = {X_jac.max():.3f}")

# By class
dup_mask = sample["is_duplicate"].values.astype(bool)
print(f"\n  mean for duplicates     = {X_jac[dup_mask].mean():.3f}")
print(f"  mean for non-duplicates = {X_jac[~dup_mask].mean():.3f}")

Jaccard stats (sample of 500):
  mean  = 0.364
  std   = 0.236
  min   = 0.000
  max   = 0.923

  mean for duplicates     = 0.465
  mean for non-duplicates = 0.301


As expected, duplicate pairs have a notably higher mean Jaccard score.

---
## `get_length_features` – Length and overlap features

Four hand-crafted features per pair:

| # | Feature | Description |
|---|---------|-------------|
| 1 | `len_diff` | |len(q1) − len(q2)| (character level) |
| 2 | `word_count_diff` | |words(q1) − words(q2)| |
| 3 | `common_word_ratio` | shared words / max word count |
| 4 | `len_ratio` | min_len / max_len |

These capture the intuition that duplicate questions tend to have similar lengths and share many words.

In [4]:
X_len = get_length_features(sample)
print("Length features shape:", X_len.shape)

feature_names = ["len_diff", "word_count_diff", "common_word_ratio", "len_ratio"]
for i, name in enumerate(feature_names):
    col = X_len[:, i]
    print(f"  {name:20s}  mean={col.mean():.3f}  std={col.std():.3f}")

Length features shape: (500, 4)
  len_diff              mean=19.152  std=24.407
  word_count_diff       mean=3.644  std=4.744
  common_word_ratio     mean=0.431  std=0.241
  len_ratio             mean=0.768  std=0.184


In [5]:
# Class comparison for common_word_ratio
print("common_word_ratio for duplicates    :", X_len[dup_mask, 2].mean().round(3))
print("common_word_ratio for non-duplicates:", X_len[~dup_mask, 2].mean().round(3))

common_word_ratio for duplicates    : 0.544
common_word_ratio for non-duplicates: 0.361
